In [ ]:
!pip install torch numpy matplotlib seaborn scikit-learn

In [ ]:
"""
4차시 실습 통합 실행 파일
모든 실습을 한 번에 실행할 수 있습니다.


Part 1: 기본 설정 및 모델 정의
Part 2: 지도학습 (분류와 회귀)
Part 3: 비지도학습과 편향-분산
Part 4: K-Fold 교차검증
Part 5: 평가 지표 계산
Part 6: 전체 ML 파이프라인


필수 라이브러리:
pip install torch numpy matplotlib seaborn scikit-learn
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression, make_blobs
# 분류, 회귀분석, 을 하기 위한 데이터를 만들어준다.
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
# 전처리 기법 => StandardScaler : 표준화 공식
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import Ridge
# test, retest 효과 => loss 감소 / variance 증가 ; 과적합 문제 발생 => 일반화 어려워
# => loss 떨어지는 것을 막아야 => 규제(regulation) L1 규제, L2 규제
# => Lasso(L1 규제, 절대값) : 특정한 점(가장자리) 추출 유리, 변수 선택 가능
# => Ridge(L2 규제, 제곱)
# https://rk1993.tistory.com/101
# 공식 정리할 것.
# training accuracy를 최소화 + Regulazaton accuracy 더해준다.
from sklearn.cluster import KMeans
# https://velog.io/@jhlee508/%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D-K-%ED%8F%89%EA%B7%A0K-Means-%EC%95%8C%EA%B3%A0%EB%A6%AC%EC%A6%98
# KMeans
# 군집 개수 k 임의로 설정 -> 중심점으로 부터 각 값이 얼마나 떨어졌는지 계산
# -> 중심점에서 가장 가까운 위치가 같은 것을 하나의 군집으로 설정
# -> 중심점(Centroid)을 재설정(갱신)
# -> 종료

# 재현성을 위한 시드 고정
torch.manual_seed(42)
np.random.seed(42)


# 한글 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False




print("=" * 70)
print("4차시 실습: 인공지능 개론 - 통합 실행")
print("=" * 70)



4차시 실습: 인공지능 개론 - 통합 실행


In [ ]:
# 모델정의
# 이진 분류용 다층 퍼셉트론 (mlp: multi layer perceptron)
class BinaryClassifier(nn.Module):
    # 생성자 => 준비물
    def __init__(self, input_dim):
        super(BinaryClassifier, self).__init__()
        # layer층
        self.layer1 = nn.Linear(input_dim, 64)
        # 입력받은 속성 정보를 64개 Node로 보낸다.
        self.layer2 = nn.Linear(64, 32)
        # 64개가 모두 32개 Node로 보낸다. ; 완전 연결(fully connected, fc)
        self.layer3 = nn.Linear(32, 1)
        # 모든 노드를 하나로 집결
        # 활성화 함수 ReLU -> 비선형성 늘려
        # ReLU는 은닉층에만 존재
        self.relu = nn.ReLU()
        # Dropout 방법
        # 연결량이 너무 많아지기 때문에, 과적합 방지를 위해 랜덤하게 연결을 끊는 방법
        # 0.3 => 30%를 끊어라.
        self.dropout = nn.Dropout(0.3)
        # 시그모이드 함수(이진분류)
        # 이진 분류 => 맞는지 아닌지 확인
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1번 layer 통과
        # ReLU를 통해 비선형성 늘린다.
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        # 2번 layer 통과
        x = self.relu(self.layer2(x))
        x = self.dropout(x)
        # 3번 레이어 통과
        # 시그모이드로 이진 분류
        x = self.sigmoid(self.layer3(x))
        return x

In [ ]:
# 회귀분석
class Regressor(nn.Module):
    # 회귀용 다층 퍼셉트론(mlp)
    def __init__(self, input_dim):
        # 명시적인 상속 요법
        super(Regressor, self).__init__()
        # functional 한 방법
        self.network = nn.Sequential(
            nn.Liear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32,1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# 지도학습 : 분류와 회귀

# 분류 데이터 생성 및 학습
X_class, y_class = \
make_classification(n_samples=1000, n_features=20, n_informative=15,
                    weights=[0.7, 0.3], n_redundant=5, random_state=42)
# n_informative : 정보값이 있는 정보, 의사소통을 하는데 의미가 있는 데이터의 개수 지정
# n_redundant : 남아있는 데이터
# random_state=42 : 랜덤 지정

In [ ]:
X_train_c, X_temp_c, y_train_c, y_temp_c = \
train_test_split(X_class, y_class, test_size=0.4, random_state=42, stratify=y_class)
# 계층적 통계분석 : stratify : 구분이 명확하게 있는 경우 분류 문제에서 사용

In [ ]:
X_val_c, X_test_c, y_val_c, y_test_c = \
train_test_split(X_temp_c, y_temp_c, test_size=0.5, random_state=42, stratify=y_temp_c)

In [ ]:
print(X_train_c.shape, X_val_c.shape, X_test_c.shape)
print(y_train_c.shape, y_val_c.shape, y_test_c.shape)
print(X_train_c[:3])
print(y_train_c[:3])

(600, 20) (200, 20) (200, 20)
(600,) (200,) (200,)
[[-2.94017183e+00  1.14911000e+00 -3.44245971e+00 -4.12591230e-01
  -6.23141406e-01 -2.08331654e+00 -2.70596986e-01 -2.66470401e+00
   3.15873651e+00  1.23236946e+00  1.08330732e+00 -6.11845789e+00
  -1.06779853e+00 -3.53338078e+00  6.86101885e-01 -2.88292219e+00
  -4.68215185e-01  4.81092604e+00 -9.54418448e-01 -2.63443743e+00]
 [ 2.71051182e-01  1.03252049e+01  3.74055379e+00 -5.24307922e+00
  -3.43153460e-02 -1.70684723e+00 -1.78586354e+00  1.76353631e+00
   6.85555902e-01 -8.03401500e-01 -2.21314118e+00 -9.16308445e-02
   4.91005141e+00 -6.52187940e+00  5.22922558e+00 -9.95806445e+00
   1.90179016e+00  1.25084516e+00 -8.46545670e-01  1.95492033e-01]
 [-4.96962193e+00 -2.95702128e+00  6.26042969e-01 -2.54818981e-01
  -3.36429822e+00  1.60926558e+00 -9.62909635e-01  2.92905602e-01
  -1.33096453e+00  3.14161288e-01  2.02676572e-01  3.68964927e+00
  -6.38065646e-01 -4.15068891e-01 -1.02117318e+00  9.74810757e-01
   2.44483199e-01  2.23

### 데이터 전처리

In [ ]:
# 표준화 (x-x_bar)/ sigma

scaler_c = StandardScaler() # 표준화 모듈
# 주의) train data 만 훈련(fit)함.
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
# >> val, test는 훈련 시키면 X / 표준화(transform)만 해준다.
X_val_c_scaled = scaler_c.transform(X_val_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

print(X_train_c_scaled[:3])

[[-6.14898829e-01 -2.71270288e-01 -1.48096434e+00  6.84758148e-04
  -5.43260121e-01 -6.08090580e-01 -3.40846443e-01 -1.31209581e+00
   1.27979769e+00  8.01054585e-01  3.93331558e-01 -1.13421669e+00
  -5.47086340e-01 -1.00117209e+00  2.18736485e-01 -1.86085928e-01
  -1.10420444e-01  1.06235358e+00 -2.39474108e-01 -9.11298620e-01]
 [-9.71745133e-02  1.76650197e+00  1.47810899e+00 -2.09251242e+00
  -3.03722787e-01 -4.40589277e-01 -1.05561705e+00  5.87445447e-01
   3.29336189e-01 -4.78452177e-02 -1.08950689e+00  1.73197518e-02
   1.86737839e+00 -2.07970477e+00  2.11839090e+00 -1.42478317e+00
   9.03437365e-01  4.09880722e-01 -1.98619714e-01  1.11122223e-01]
 [-9.42093690e-01 -1.18313536e+00  1.95072850e-01  6.90522660e-02
  -1.65837618e+00  1.03483834e+00 -6.67419169e-01 -4.33973939e-02
  -4.45627450e-01  4.18169304e-01 -2.80177669e-03  7.39803043e-01
  -3.73516427e-01  1.24209493e-01 -4.95140757e-01  4.89315780e-01
   1.94462759e-01  5.89471851e-01  1.23753102e-01  1.68004220e+00]]


In [ ]:
# 데이터타입 변경 (torch.tensor)

X_train_c_t = torch.FloatTensor(X_train_c_scaled)
# 차원 확장 : 1차원 -> 2차원으로
y_train_c_t = torch.FloatTensor(y_train_c).unsqueeze(1)
X_val_c_t = torch.FloatTensor(X_val_c_scaled)
y_val_c_t = torch.FloatTensor(y_val_c).unsqueeze(1)

# 모델 설정
model_class = BinaryClassifier(input_dim=20)
criterion = nn.BCELoss()    # Binary cross entropy loss
optimizer = optim.Adam(model_class.parameters(), lr=0.001)

In [ ]:
print(model_class)

BinaryClassifier(
  (layer1): Linear(in_features=20, out_features=64, bias=True)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (layer3): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (sigmoid): Sigmoid()
)


In [ ]:
print(criterion)

BCELoss()


In [ ]:
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
